CSV and Excel file - Structured data 

In [7]:
import pandas as pd
import os 

In [3]:
os.makedirs("data/structured_files" , exist_ok=True)

In [8]:
# Create Sample Data
data = {
    "Product": [
        "Laptop",
        "Wireless Mouse",
        "Mechanical Keyboard",
        "Monitor",
        "Headphones"
    ],

    "Category": [
        "Electronics",
        "Accessories",
        "Accessories",
        "Electronics",
        "Accessories"
    ],

    "Price": [
        1200,
        25,
        80,
        350,
        120
    ],

    "Stock": [
        15,
        50,
        30,
        12,
        25
    ],

    "Description": [
        "High-performance laptop with 16GB RAM and 512GB SSD.",
        "Ergonomic wireless mouse with long battery life.",
        "RGB mechanical keyboard with tactile switches.",
        "27-inch high-resolution monitor for work and gaming.",
        "Noise-cancelling wireless headphones with clear audio."
    ]
}
## Save as CSV
df = pd.DataFrame(data)
df.to_csv('data/structured_files/products.csv' , index=False)

### For Excel File 

In [15]:
with pd.ExcelWriter("data/structured_files/inventory.xlsx") as writer:
    
 df.to_excel(writer , sheet_name="Products" ,index=False)

summary_data = {
    'Category': ['Eelectronics' , 'Accessories'],
    'Total_Items':[3 , 2],
    'Total_Values':[109.09 , 23.5467],
}
pd.DataFrame(summary_data).to_excel(writer , sheet_name="Summary" , index=False)

### CSV Processing

In [16]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

/Users/macbookpro/Documents/Projects/Rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
## Method 1 CSV Loader - Each row becomes a document 
print("CSV Loader Row Based Documents")
csv_loader = CSVLoader(
    file_path="data/structured_files/products.csv",
    encoding="utf-8",
    csv_args={
        'delimiter': ',',
        'quotechar': '"'
    }
)
csv_docs = csv_loader.load()
print(csv_docs)
print(f"Loaded {len(csv_docs)} Documents (one per row)")
print("\nFirst Document:")
print(f"Content: {csv_docs[4].page_content}")
print(f"Meta data: {csv_docs[4].metadata}")


CSV Loader Row Based Documents
[Document(metadata={'source': 'data/structured_files/products.csv', 'row': 0}, page_content='Product: Laptop\nCategory: Electronics\nPrice: 1200\nStock: 15\nDescription: High-performance laptop with 16GB RAM and 512GB SSD.'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 1}, page_content='Product: Wireless Mouse\nCategory: Accessories\nPrice: 25\nStock: 50\nDescription: Ergonomic wireless mouse with long battery life.'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 2}, page_content='Product: Mechanical Keyboard\nCategory: Accessories\nPrice: 80\nStock: 30\nDescription: RGB mechanical keyboard with tactile switches.'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 3}, page_content='Product: Monitor\nCategory: Electronics\nPrice: 350\nStock: 12\nDescription: 27-inch high-resolution monitor for work and gaming.'), Document(metadata={'source': 'data/structured_files/products.

In [27]:
from typing import List
from langchain_core.documents import Document

## Method 2 - Custom CSV Processign for better control 
print("\nCustom CSV Processing")
def process_csv_intelligently(filePath : str) -> List[Document]:
    """Process CSV with intelligency creation"""
    df = pd.read_csv(filePath)
    documents = []
    ## Strategy 1 One document per row with structured row 
    for idx , row in df.iterrows():
        ## Create Structured Content 
        content = f"""Product Information:
        Name:{row['Product']},
        Category:{row['Category']},
        Price:{row['Price']},
        Stock:{row['Stock']},
        Description:{row['Description']},
        """
        ## Create document with rich metadata
        doc = Document(
            page_content=content,
            metadata= {
                'source' : filePath,
                'row_index' : idx,
                'product_name':row['Product'],
                'category': row['Category'],
                'price':row['Price'],
                'data_type': 'product_info'
            }
        )
        documents.append(doc)
        
    return documents


Custom CSV Processing


In [28]:
process_csv_intelligently("data/structured_files/products.csv")

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 1200, 'data_type': 'product_info'}, page_content='Product Information:\n        Name:Laptop,\n        Category:Electronics,\n        Price:1200,\n        Stock:15,\n        Description:High-performance laptop with 16GB RAM and 512GB SSD.,\n        '),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'Wireless Mouse', 'category': 'Accessories', 'price': 25, 'data_type': 'product_info'}, page_content='Product Information:\n        Name:Wireless Mouse,\n        Category:Accessories,\n        Price:25,\n        Stock:50,\n        Description:Ergonomic wireless mouse with long battery life.,\n        '),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product_name': 'Mechanical Keyboard', 'category': 'Accessories', 'price': 80, 'data_type': 'product_info'}, pag

In [41]:
print("CSV Processign Strategies:")
print("\n 1. Row Based Loader")
print("    Simple one-row-one document")
print("    Good for record lookup")
print("    Losses table context")
print("\n 2. Intelligent Processing")
print("    Prederves relationships")
print("    Create summaries")
print("    Rich metadata")
print("    Better for QA")

CSV Processign Strategies:

 1. Row Based Loader
    Simple one-row-one document
    Good for record lookup
    Losses table context

 2. Intelligent Processing
    Prederves relationships
    Create summaries
    Rich metadata
    Better for QA


### Excel Processing

In [48]:
from typing import List
from langchain_core.documents import Document

## Method 2 - Using pandas for fullcontrol 
print("\nPandas Bases Excel Processing")
def process_excel_with_pandas(filePath : str) -> List[Document]:
    """Process excek with sheet awareness"""
    documents = []
    ## Read excel sheet 
    excel_file = pd.ExcelFile(filePath)
    
    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filePath , sheet_name=sheet_name)
        ## Create document for each sheet 
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {", ".join(df.columns)}\n"
        sheet_content += f"Row: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)
        doc = Document(
                    page_content=sheet_content,
                    metadata= {
                        'source' : filePath,
                        'sheet_name' : sheet_name,
                        'num_rows':len(df),
                        'num_columns':len(df.columns),
                        'data_type': 'excel_sheet'
                    }
                )
    documents.append(doc)
    
    return documents
        
        
        


Pandas Bases Excel Processing


In [50]:
excel_docs= process_excel_with_pandas("data/structured_files/inventory.xlsx")
print(f"Processed {len(excel_docs)} Shetts")

Processed 1 Shetts


In [51]:
excel_docs

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nRow: 5\n\n            Product    Category  Price  Stock                                            Description\n             Laptop Electronics   1200     15   High-performance laptop with 16GB RAM and 512GB SSD.\n     Wireless Mouse Accessories     25     50       Ergonomic wireless mouse with long battery life.\nMechanical Keyboard Accessories     80     30         RGB mechanical keyboard with tactile switches.\n            Monitor Electronics    350     12   27-inch high-resolution monitor for work and gaming.\n         Headphones Accessories    120     25 Noise-cancelling wireless headphones with clear audio.')]

In [61]:
from langchain_community.document_loaders import UnstructuredExcelLoader
## Method 2 Unstructured excel loader 
print("Unstructured Excel Loader")
try:
    excel_loader = UnstructuredExcelLoader('data/structured_files/inventory.xlsx' , mode='elements')
    
    unstructured_docs = excel_loader.load()
    print("Handle complex excel features")
    print("Preserves formating info")
    print("Require unstructired libraries")
except Exception as e:
    print(f"Error{e}")

Unstructured Excel Loader
Handle complex excel features
Preserves formating info
Require unstructired libraries


In [62]:
unstructured_docs

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-09-16T01:19:30', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>1200</td><td>15</td><td>High-performance laptop with 16GB RAM and 512GB SSD.</td></tr><tr><td>Wireless Mouse</td><td>Accessories</td><td>25</td><td>50</td><td>Ergonomic wireless mouse with long battery life.</td></tr><tr><td>Mechanical Keyboard</td><td>Accessories</td><td>80</td><td>30</td><td>RGB mechanical keyboard with tactile switches.</td></tr><tr><td>Monitor</td><td>Electronics</td><td>350</td><td>12</td><td>27-inch high-resolution monitor for work and gaming.</td></tr><tr><td>Headphones</td><td>Accessories</td><td>120</td><td>25</td><td>Noise-cancelling wireless headphones with clear audio.</td></tr></t